In [1]:
import io, os, warnings, torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from torch.utils.data import Dataset, DataLoader
from transformers import HubertModel, AutoFeatureExtractor, AutoModel, AutoTokenizer
from collections import Counter
from tqdm import tqdm

warnings.filterwarnings('ignore')
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


# CONFIG

PARQUET_DIR = "/kaggle/input/datasets/thevinu/dataset" 
SAVE_PATH = "/kaggle/working/multimodal_emotion_final.pt" 
AUDIO_BRAIN = "/kaggle/input/datasets/thevinu/hubert-brain"
TEXT_BRAIN = "/kaggle/input/datasets/thevinu/roberta-base-local"

SR = 16000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4    
GRAD_ACCUM = 8    # Effective Batch Size = 32
EPOCHS = 15


# DATA ENGINE

def process_soft_labels(row):
    label = np.zeros(4, dtype=np.float32)
    label[0], label[1], label[2] = row['angry'], row['sad'], row['neutral']
    label[3] = row['happy'] + row['excited']
    total = label.sum()
    return label / total if total > 0 else label

class MultimodalIEMOCAP(Dataset):
    def __init__(self, records, audio_extractor, text_tokenizer):
        self.records = records
        self.extractor = audio_extractor
        self.tokenizer = text_tokenizer
    def __len__(self): return len(self.records)
    def __getitem__(self, idx):
        audio, text, label = self.records[idx]
        if len(audio) > 128000: audio = audio[:128000] # ~8 seconds
        else: audio = np.pad(audio, (0, max(0, 128000 - len(audio))))
        
        audio_feat = self.extractor(audio, sampling_rate=SR, return_tensors="pt").input_values.squeeze(0)
        text_tokens = self.tokenizer(text, max_length=128, padding='max_length', truncation=True, return_tensors="pt")
        
        return audio_feat, text_tokens['input_ids'].squeeze(0), text_tokens['attention_mask'].squeeze(0), torch.FloatTensor(label)

def load_and_prepare_data():
    train_recs, test_recs = [], []
    pq_files = sorted([f for f in os.listdir(PARQUET_DIR) if f.endswith('.parquet')])
    for pq_file in pq_files:
        df = pd.read_parquet(os.path.join(PARQUET_DIR, pq_file))
        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Loading {pq_file}"):
            try:
                # TRIPLE CHECK: Session 05 is the "Unseen" Test Set
                sess_id = str(row['file'])[:5] 
                audio, _ = sf.read(io.BytesIO(row['audio']['bytes']))
                if audio.ndim > 1: audio = audio.mean(axis=1)
                audio = librosa.resample(audio.astype(np.float32), orig_sr=16000, target_sr=SR)
                label = process_soft_labels(row)
                
                if sess_id == 'Ses05': test_recs.append((audio, str(row['transcription']), label))
                else: train_recs.append((audio, str(row['transcription']), label))
            except: continue
    return train_recs, test_recs


# GATED RESIDUAL ARCHITECTURE

class GatedEmotionModel(nn.Module):
    def __init__(self, audio_dim=1024, text_dim=768, hidden_dim=512):
        super().__init__()
        self.hubert = HubertModel.from_pretrained(AUDIO_BRAIN)
        self.roberta = AutoModel.from_pretrained(TEXT_BRAIN)
        
        self.audio_proj = nn.Linear(audio_dim, hidden_dim)
        self.text_proj = nn.Linear(text_dim, hidden_dim)
        
        # Text-driven attention over Audio
        self.cross_attn = nn.MultiheadAttention(embed_dim=hidden_dim, num_heads=8, batch_first=True, dropout=0.3)
        
        # THE GATE: Learns when to trust text over audio
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU(), nn.Linear(256, 4)
        )

    def forward(self, audio_x, input_ids, attn_mask):
        a_feat = self.hubert(audio_x).last_hidden_state 
        t_feat = self.roberta(input_ids=input_ids, attention_mask=attn_mask).last_hidden_state
        
        a_p = self.audio_proj(a_feat) 
        t_p = self.text_proj(t_feat)  
        
        # Use Text to query Audio
        attn_out, _ = self.cross_attn(query=t_p, key=a_p, value=a_p)
        
        a_mean = torch.mean(a_p, dim=1)    
        t_attn_mean = torch.mean(attn_out, dim=1) 
        
        # Gating logic
        g = self.gate(torch.cat([a_mean, t_attn_mean], dim=-1))
        fused = a_mean + (g * t_attn_mean) # Residual Connection
        
        return self.classifier(fused)


# MAIN ENGINE

def main():
    train_recs, test_recs = load_and_prepare_data()
    extractor = AutoFeatureExtractor.from_pretrained(AUDIO_BRAIN)
    tokenizer = AutoTokenizer.from_pretrained(TEXT_BRAIN)
    
    train_loader = DataLoader(MultimodalIEMOCAP(train_recs, extractor, tokenizer), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(MultimodalIEMOCAP(test_recs, extractor, tokenizer), batch_size=BATCH_SIZE, num_workers=2)

    model = GatedEmotionModel().to(DEVICE)
    
    # Partial Unfreeze for fine-tuning
    for p in model.hubert.encoder.layers[-2:].parameters(): p.requires_grad = True
    for p in model.roberta.encoder.layer[-2:].parameters(): p.requires_grad = True

    optimizer = optim.AdamW([
        {'params': model.hubert.parameters(), 'lr': 1e-5},
        {'params': model.roberta.parameters(), 'lr': 1e-5},
        {'params': model.classifier.parameters(), 'lr': 2e-4}
    ], weight_decay=0.01)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.cuda.amp.GradScaler()
    best_acc = 0

    print(f"\n🚀 STARTING FINAL RUN | Train: {len(train_recs)} | Test: {len(test_recs)}")

    for epoch in range(EPOCHS):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
        
        optimizer.zero_grad()
        for i, (a, ids, mask, labels) in enumerate(pbar):
            a, ids, mask, labels = a.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            
            with torch.cuda.amp.autocast():
                logits = model(a, ids, mask)
                loss = criterion(logits, labels) / GRAD_ACCUM
            
            scaler.scale(loss).backward()
            
            if (i + 1) % GRAD_ACCUM == 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
            
            # Metrics
            train_loss += loss.item() * GRAD_ACCUM
            train_correct += (torch.argmax(logits, 1) == torch.argmax(labels, 1)).sum().item()
            train_total += labels.size(0)
            pbar.set_postfix({'Loss': f"{train_loss/(i+1):.4f}", 'Acc': f"{100*train_correct/train_total:.2f}%"})

        # VALIDATION
        model.eval()
        val_loss, val_preds, val_targets = 0, [], []
        with torch.no_grad():
            for a, ids, mask, labels in test_loader:
                a, ids, mask, labels = a.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
                out = model(a, ids, mask)
                loss = criterion(out, labels)
                val_loss += loss.item()
                val_preds.extend(torch.argmax(out, dim=1).cpu().numpy())
                val_targets.extend(torch.argmax(labels, dim=1).cpu().numpy())
        
        v_acc = (np.array(val_preds) == np.array(val_targets)).mean() * 100
        t_acc = (train_correct / train_total) * 100
        v_loss_avg = val_loss / len(test_loader)
        
        print(f"\n📊 EPOCH {epoch+1} SUMMARY")
        print(f"Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {t_acc:.2f}%")
        print(f"Val Loss: {v_loss_avg:.4f} | Val Acc: {v_acc:.2f}%")
        print(f"Dist: {Counter(val_preds)}\n")
        
        if v_acc > best_acc:
            best_acc = v_acc
            torch.save(model.state_dict(), SAVE_PATH)
            print(f"⭐ New Best Model Saved: {best_acc:.2f}%")

if __name__ == "__main__":
    main()

Loading 0002.parquet: 100%|██████████| 3346/3346 [00:03<00:00, 1068.06it/s]


Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


🚀 STARTING FINAL RUN | Train: 7869 | Test: 2170


Epoch 1: 100%|██████████| 1968/1968 [15:37<00:00,  2.10it/s, Loss=1.3482, Acc=33.38%]


...


📊 EPOCH 13 SUMMARY
Train Loss: 0.9489 | Train Acc: 72.47%
Val Loss: 1.1193 | Val Acc: 58.66%
Dist: Counter({np.int64(2): 959, np.int64(0): 514, np.int64(3): 506, np.int64(1): 191})



Epoch 14: 100%|██████████| 1968/1968 [15:46<00:00,  2.08it/s, Loss=0.9309, Acc=74.09%]



📊 EPOCH 14 SUMMARY
Train Loss: 0.9309 | Train Acc: 74.09%
Val Loss: 1.0891 | Val Acc: 61.61%
Dist: Counter({np.int64(2): 862, np.int64(0): 544, np.int64(3): 527, np.int64(1): 237})



Epoch 15: 100%|██████████| 1968/1968 [15:46<00:00,  2.08it/s, Loss=0.9188, Acc=75.42%]



📊 EPOCH 15 SUMMARY
Train Loss: 0.9188 | Train Acc: 75.42%
Val Loss: 1.0790 | Val Acc: 63.04%
Dist: Counter({np.int64(2): 775, np.int64(3): 659, np.int64(0): 529, np.int64(1): 207})

⭐ New Best Model Saved: 63.04%
